# PHASE 1 — MACHINE LEARNING FOUNDATIONS


# Day 03 — Dataset Splitting


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Use `train_test_split()` effectively.
- Differentiate between the Training Set, Validation Set, and Test Set.
- Explain Data Leakage and why evaluating on training data is dangerous.
- Use `random_state` for reproducibility.
- Apply stratification to handle imbalanced datasets.


## 2. Prerequisites
- Basic understanding of the Scikit-learn API (`fit`, `predict`).
- A conceptual grasp of features (`X`) and targets (`y`).


## 3. Concept
Machine learning models are prone to **memorizing** data rather than learning general patterns. If a model simply memorizes the data, it will perform perfectly on the data it has seen but fail miserably on new, unseen data.

To prevent this, we split our data:
- **Training Set**: Used to fit the model (study).
- **Validation Set**: Used to tune the model (practice exams).
- **Test Set**: Used exactly once to evaluate the final model (final exam).


## 4. Why Does This Matter?
If you evaluate a model on the same data you used to train it, you get an overly optimistic score. This is called **Data Leakage** (specifically, train/test contamination). 

**Rule of Thumb**: NEVER touch the test set until you are 100% finished training and tuning your model.


## 5. Intuition
Imagine taking a math test where the teacher gave you the exact test questions the night before to study. If you score 100%, does it mean you are good at math, or does it mean you just memorized those specific questions? 

By withholding a portion of the data (the test set), we evaluate the model's true ability to **generalize** to new situations.


## 6. Mathematical Foundation
When we evaluate a model, we want to estimate the expected out-of-sample error:

$$ Err_{T} = E[L(Y, \hat{f}(X)) | T] $$

Where $T$ is the training set. If we estimate this error using the training set itself, the estimate is downward-biased (too low) because the model parameters were chosen specifically to minimize the error on $T$.

The test set provides an unbiased estimate of this error because it is independent of $T$.


## 7. Scikit-learn API
Scikit-learn provides `train_test_split` inside the `model_selection` module.


In [ ]:
from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 8. Simple Example
Let's generate an imbalanced classification dataset and see what happens when we split it.


In [ ]:
from sklearn.datasets import make_classification
import numpy as np

# Create imbalanced data (90% class 0, 10% class 1)
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, weights=[0.9], random_state=42)
print('Original Target Distribution:')
print(f'Class 0: {np.mean(y==0)*100:.1f}%')
print(f'Class 1: {np.mean(y==1)*100:.1f}%\n')

# Basic split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Test Set Distribution (Without Stratification):')
print(f'Class 0: {np.mean(y_test==0)*100:.1f}%')
print(f'Class 1: {np.mean(y_test==1)*100:.1f}%')


## 9. Code Walkthrough
- `make_classification(..., weights=[0.9])`: Creates a dataset where 90% of samples belong to Class 0.
- `train_test_split`: Shuffles the data and splits 80% to train, 20% to test.
- Note that in the output, the test set distribution might not perfectly match the original 90/10 split due to random chance.


## 10. Experiment
Change the code to use **stratification**. Stratification ensures the train and test sets have the exact same proportion of classes as the original dataset.


In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Test Set Distribution (With Stratification):')
print(f'Class 0: {np.mean(y_test_s==0)*100:.1f}%')
print(f'Class 1: {np.mean(y_test_s==1)*100:.1f}%')


> Notice how it perfectly matches the original 90/10 split!


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y, test_size=0.2, random_state=99)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.2, random_state=99)


> **Question:** Are `X_train1` and `X_train2` identical arrays? What happens if you remove `random_state=99`?

**Think before running the next cell!**


In [ ]:
print('Are X_train1 and X_train2 identical?', np.array_equal(X_train1, X_train2))
print('\nWhy? `random_state` seeds the random number generator. Using the same seed guarantees the exact same shuffle and split every time. If you remove it, you get a different split every time you run the code.')


## 12. Coding Exercise
Create a **Train / Validation / Test** split.
1. Split `X` and `y` into `X_temp, X_test, y_temp, y_test` (Test size = 20%).
2. Split `X_temp` and `y_temp` into `X_train, X_val, y_train, y_val` (Validation size = 25% of the temporary set).


In [ ]:
# YOUR CODE HERE
X_temp, X_test_final, y_temp, y_test_final = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# 25% of 80% is 20% of the total dataset.
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f'Train shape: {X_train_final.shape}')
print(f'Val shape: {X_val_final.shape}')
print(f'Test shape: {X_test_final.shape}')


## 13. Debugging Challenge
The junior data scientist tried to split the data, but the model crashed during training. Find the bug!


In [ ]:
from sklearn.linear_model import LogisticRegression

# Buggy code
try:
    X_train_bug, X_test_bug, y_train_bug = train_test_split(X, y, test_size=0.2)
    model = LogisticRegression()
    model.fit(X_train_bug, y_train_bug)
except Exception as e:
    print('Error:', type(e).__name__)
    print('Message:', e)


> **Hint:** Look at how many variables `train_test_split` unpacks into. Count them carefully!


## 14. Model Evaluation
Let's prove why we split data. We will train a Decision Tree that is highly prone to memorizing data (overfitting).


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_s, y_train_s)

train_score = model.score(X_train_s, y_train_s)
test_score = model.score(X_test_s, y_test_s)

print(f'Training Accuracy (Memorized): {train_score*100:.2f}%')
print(f'Test Accuracy (Generalization): {test_score*100:.2f}%')


> If we didn't have a test set, we would think our model is 100% perfect!


## 15. Real-World Example
In real-world finance (e.g., predicting stock prices), random splitting is actually a bad idea! Since time matters, you must use **Time-Series Splitting** (e.g., train on 2018-2022, test on 2023). Random splitting would cause future data to leak into the training set!


## 16. Mini Project
Write a function `evaluate_split_impact(test_size)` that:
1. Splits `X` and `y` (the imbalanced dataset from above) using the given `test_size` (no stratify).
2. Trains a Logistic Regression model.
3. Returns the test accuracy.


In [ ]:
from sklearn.linear_model import LogisticRegression
def evaluate_split_impact(test_size):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=42)
    model = LogisticRegression(random_state=42)
    model.fit(X_tr, y_tr)
    return model.score(X_te, y_te)

print('Test Size 0.1:', evaluate_split_impact(0.1))
print('Test Size 0.9:', evaluate_split_impact(0.9))


## 17. Common Mistakes
- **Forgetting `random_state`**: Causes your results to change every time you run the notebook, making debugging impossible.
- **Not using `stratify` on imbalanced data**: You might randomly get a test set that contains zero instances of the minority class!
- **Leaking information**: E.g., scaling the data *before* splitting it. (We will cover this in depth tomorrow).


## 18. Interview Questions
- **Beginner**: Why do we need a test set?
- **Intermediate**: What is the difference between a validation set and a test set?
- **Advanced**: When would you avoid using `train_test_split` with `shuffle=True`?


## 19. Knowledge Check
- What parameter ensures reproducible splits? (`random_state`)
- What parameter ensures class proportions are maintained? (`stratify`)


## 20. Summary
- Never evaluate on training data.
- **Train** = Study. **Validation** = Practice Exam. **Test** = Final Exam.
- Use `stratify=y` for classification problems.
- Memorization is not Learning. Evaluation on unseen data proves generalization.


## 21. Homework
Load the `load_digits()` dataset. Split it into 70% train and 30% test using stratification. Verify that the distributions of the digits (0-9) are equal in both sets using `np.bincount()`.
